In [1]:
import json
import os
import re
from bs4 import BeautifulSoup

In [11]:

#Läs in JSON-filen jag skapade  
with open("matematik.json", "r", encoding="utf-8") as f:
    data = json.load(f)
    
subject = data.get("subject", {})
chunks = []

# Skriv ut nycklar för att dubbelkolla struktur
print(data.keys())
print(subject.keys())

dict_keys(['apiPublisher', 'apiName', 'apiVersion', 'apiReleased', 'apiStatus', 'apiDocumentation', 'dataOrigin', 'calledMethod', 'totalElements', 'processingTime', 'startedCaching', 'codeParam', 'dateParam', 'subject'])
dict_keys(['centralCntHeading', 'centralContents', 'code', 'description', 'designation', 'gradeScale', 'knowledgeReqsHeading', 'knowledgeRequirements', 'modifiedDate', 'name', 'nameHeading', 'purpose', 'purposeHeading', 'schoolTypes', 'skolfsAndring', 'skolfsGrund', 'startDate', 'typeOfSyllabus', 'version', 'versionInfo'])


In [12]:
#Grundläggande metadata
subject_name = subject.get("name", "Okänt ämne")
subject_code = subject.get("code", "Okänd kod")
subject_version = subject.get("version", "Okänd version")

In [3]:
centralt = subject.get('centralContents', []) 
for section in centralt: 
    print(section.get('heading', ''), '\n', section.get('text', '')[:100], '\n---') #Visa första 100 tecken för översikt/snabbkoll

 
 <h3>I årskurs 1–3</h3><h4>Taluppfattning och tals användning</h4> <ul> <li>Naturliga tal och deras e 
---
 
 <h3>I årskurs 4–6</h3><h4>Taluppfattning och tals användning</h4> <ul> <li>Rationella tal, däribland 
---
 
 <h3>I årskurs 7–9</h3><h4>Taluppfattning och tals användning</h4> <ul> <li>Reella tal och deras egen 
---


In [13]:
#Funktion för att ta bort HTML-taggar och returnera ren text
def clean_html(html_text):
    if not html_text:
        return ""

    soup = BeautifulSoup(str(html_text), "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    return text

In [14]:
#Funktion för att dela upp text utifrån h4-taggar 
def split_by_h4(html_text):
    if not html_text:
        return []

    soup = BeautifulSoup(str(html_text), "html.parser")

    results = []
    current_h4 = None
    current_content = []

    for tag in soup.find_all(["h4", "ul", "p"]):
        if tag.name == "h4":
            if current_h4 and current_content:
                text = " ".join(current_content).strip()
                if text:
                    results.append({
                        "subheading": current_h4,
                        "text": text
                    })

            current_h4 = tag.get_text(" ", strip=True)
            current_content = []

        elif tag.name == "ul" and current_h4:
            items = [li.get_text(" ", strip=True) for li in tag.find_all("li")]
            if items:
                current_content.extend(items)

        elif tag.name == "p" and current_h4:
            p_text = tag.get_text(" ", strip=True)
            if p_text:
                current_content.append(p_text)

    if current_h4 and current_content:
        text = " ".join(current_content).strip()
        if text:
            results.append({
                "subheading": current_h4,
                "text": text
            })

    return results

In [15]:
# 1. Syfte
syfte_text = clean_html(subject.get("purpose"))
if syfte_text:
    chunks.append({
        "subject": subject_name,
        "subject_code": subject_code,
        "version": subject_version,
        "section": "syfte",
        "year": None,
        "heading": "Syfte",
        "text": syfte_text
    })

In [19]:
central_contents = subject.get("centralContents", [])
print(f"Hittade {len(central_contents)} centralt innehåll-objekt")

for i, item in enumerate(central_contents):
    year = item.get("year")
    html_text = item.get("text", "")

    split_parts = split_by_h4(html_text)

    # Om det finns h4-rubriker: skapa ett chunk per underrubrik
    if split_parts:
        for part in split_parts:
            chunks.append({
                "subject": subject_name,
                "subject_code": subject_code,
                "version": subject_version,
                "section": "centralt_innehall",
                "year": year,
                "subheading": part["subheading"],
                "heading": f"{part['subheading']} ({year})",
                "text": part["text"]
            })
    else:
        # Fallback om texten saknar h4-rubriker
        text = clean_html(html_text)
        if text:
            chunks.append({
                "subject": subject_name,
                "subject_code": subject_code,
                "version": subject_version,
                "section": "centralt_innehall",
                "year": year,
                "subheading": None,
                "heading": f"Centralt innehåll {year}",
                "text": text
            })

Hittade 3 centralt innehåll-objekt


In [20]:
# 3. Kunskapskrav / betygskriterier
knowledge_reqs = subject.get("knowledgeRequirements", [])
print(f"Hittade {len(knowledge_reqs)} kunskapskrav-objekt")
for i, item in enumerate(knowledge_reqs):
    text = clean_html(item.get("text"))
    year = item.get("year")
    
    if text:
        chunks.append({
            "subject": subject_name,
            "subject_code": subject_code,
            "version": subject_version,
            "section": "kunskapskrav",
            "year": year,
            "heading": f"Kunskapskrav {year}",
            "text": text
        })

Hittade 11 kunskapskrav-objekt


In [23]:
#Spara resultatet 
os.makedirs("data", exist_ok=True)

with open("data/matematik_chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print(f"Klart! Skapade {len(chunks)} chunkar.")

Klart! Skapade 59 chunkar.
